# [Chapter 8 — Parameter Estimation, §8.1] The Inverse Problem: From Data to Parameters

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 8 — Parameter Estimation
**Considerations developed:** 8 (parameter estimation)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Frames parameter estimation as an inverse problem: given observed incidence $J(t)$, infer the parameters $(c_I, \beta, \tau_R)$ or equivalently $(\mathcal{R}_0, \tau_R)$. Establishes the notation used throughout Chapter 8.

## What you should already know
Chapter 7 simulations notebooks; basic familiarity with least-squares fitting.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import (
    book_style,
    BOOK_COLORS,
    integrate_sir_i,
    baseline_central_comparison,
)
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance

set_seed_chapter_08()
book_style()


## The forward and inverse problems

The **forward problem** is: given parameters $(c_I, \beta, \tau_R)$ and initial conditions $(S_0, I_0, R_0)$, compute the SIR_I trajectory $(S(t), I(t), R(t))$. We have done this in Chapter 7.

The **inverse problem** is: given observed incidence data $\{J(t_i)\}$, infer the parameters that produced them.

This is harder because the data are noisy, the parameters are not always identifiable, and different parameter sets can produce nearly identical trajectories. Chapter 8 develops the tools to do this carefully.

In [ ]:
# Generate a synthetic outbreak as our 'true' system
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']

# Forward problem: simulate
result = integrate_sir_i(params)
t = result['t']
S = result['S']
I = result['I']

# Compute true incidence J(t) = alpha(t) * I(t)
N = params['N']
alpha_t = params['c_I'] * params['beta'] * (S / N)
J_true = alpha_t * I

# Add observation noise (5% of peak)
sigma = params['noise_sigma'] * J_true.max()
J_obs = np.maximum(J_true + sigma * np.random.randn(len(t)), 0)

print(f"True R_0 = {true_R0:.2f}")
print(f"Peak true incidence: {J_true.max():.5f}")
print(f"Observation noise: sigma = {sigma:.5f}")
print(f"Number of observations: {len(t)}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, J_obs, '.', color=BOOK_COLORS['neutral'], alpha=0.3, ms=3, label='Observed J(t)')
ax.plot(t, J_true, color=BOOK_COLORS['infectious'], lw=1.5, label='True J(t)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Incidence J(t)')
ax.set_title('The data: noisy observations of incidence')
ax.legend()
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

print('\nThe inverse problem: given the noisy J(t) curve, recover R_0 = 2.0.')
print('The next notebooks compare two estimators that approach this differently.')


## What's next

The next two notebooks set up the two estimators side-by-side:
- `02_alpha_estimator_baseline.ipynb` — the infected-viewpoint estimator $\hat\alpha = J/I$
- `03_lambda_estimator_baseline.ipynb` — the susceptible-viewpoint estimator $\hat\lambda = J/S^*$

Then `04_central_comparison_alpha_vs_lambda.ipynb` runs them head-to-head.